# OSFI Regulation Assistant: a RAG project
## Part 5: Run the packaged code as an API

Parts 1–4 were built and evaluated in notebooks. Part 5 runs the **packaged version** from your GitHub repository, the way it would run in production:

1. Get the code from GitHub
2. Run the **automated tests**
3. Build the **final search index** with all the evaluated fixes
4. Start the **REST API** (FastAPI) and call it like any other application would
5. Open the API's **interactive documentation** in your browser

**Before running:** upload the repository to GitHub (see the instructions in the chat), and set `REPO_URL` below.

**Runtime: T4 GPU.**

## Step 1: Get the code

In [1]:
REPO_URL = "https://github.com/RubyaAfrin/osfi-rag-assistant"   # change if your repository has a different name

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/osfi_rag_project"
except Exception as e:
    print("Drive not available, using a local folder:", e)
    PROJECT_DIR = "osfi_rag_project"

PROJECT_DIR = os.path.abspath(PROJECT_DIR)
REPO_DIR = "/content/osfi-rag-assistant" if os.path.exists("/content") else os.path.abspath("osfi-rag-assistant")
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -3

Mounted at /content/drive
f4ca795 (HEAD -> main, origin/main, origin/HEAD) Add files via upload
be54fc4 Delete notebooks/osfi_rag_part4_evaluation.ipynb
a431ba9 Delete notebooks/osfi_rag_part3_generation.ipynb


## Step 2: Install dependencies and run the tests

In [2]:
!pip -q install -r requirements.txt -r requirements-test.txt
!python -m pytest -q

..........................                                               [100%]
26 passed in 0.52s


These are the same tests GitHub Actions runs every time you push a change. They use small stand-in models, so they finish in seconds, and several of them reproduce the exact bugs found in Part 4, so those bugs can't return unnoticed.

## Step 3: Build the final index

`scripts/build_index.py` takes the chunks from Part 2, applies the definition labels, and embeds everything. The final settings, stemming and a re-ranker pool of 50, are the defaults in the package.

In [3]:
CHUNKS = os.path.join(PROJECT_DIR, "index", "chunks.jsonl")
FINAL_INDEX = os.path.join(PROJECT_DIR, "index_final")
!python scripts/build_index.py --chunks "{CHUNKS}" --out "{FINAL_INDEX}"

modules.json: 100% 349/349 [00:00<00:00, 1.09MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 727kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 113MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 338kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.49MB/s]

model.safetensors: downloading bytes:  56% 74.4M/133M [00:01<00:00, 89.7MB/s, 6.51MB/s  ]
model.safetensors: reconstructing file:  85% 114M/133M [00:01<00:00, 85.5MB/s]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 50.0MB/s, 7.88MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 77.0MB/s, 12.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5788.98it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.73MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 34.4MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 66.2MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 595kB/s]
config.json: 100% 190/190 [00:00<00:00, 861kB/s]
config.json: 100% 794/794 [0

## Step 4: Start Ollama and the API

In [4]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests, json
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b 2>&1 | tail -1

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
success 


In [5]:
subprocess.run(["pkill", "-f", "uvicorn osfi_rag.api"])   # stop a server left over from an earlier run
time.sleep(1)
env = dict(os.environ, OSFI_INDEX_DIR=FINAL_INDEX, OSFI_LLM_MODEL="qwen2.5:7b")
api_log = open("/tmp/api.log", "w")
api = subprocess.Popen(["uvicorn", "osfi_rag.api:app", "--host", "0.0.0.0", "--port", "8000"],
                       env=env, stdout=api_log, stderr=subprocess.STDOUT)
BASE = "http://localhost:8000"
for _ in range(60):
    try:
        print(requests.get(f"{BASE}/health", timeout=2).json()); break
    except Exception:
        time.sleep(2)
else:
    print("The API did not start. Its log:"); print(open("/tmp/api.log").read()[-3000:])

{'status': 'ok', 'version': '1.0.0', 'loaded': False}


## Step 5: Call the API

This is exactly how another application, such as a website, a chatbot, or an internal tool, would use the assistant: by sending an HTTP request and receiving JSON. The first call loads the models, so it takes longer.

In [6]:
def ask(question):
    r = requests.post(f"{BASE}/ask", json={"question": question}, timeout=300)
    r.raise_for_status()
    body = r.json()
    print(f"Q: {question}\n[{body['status'].upper()}] {body['answer']}")
    for s in body["sources"]:
        if s["cited"]:
            print(f"   [{s['n']}] {s['doc_id']} › {s['section']}")
    print(f"   ({body['seconds']}s)\n")
    return body

examples = [ask(q) for q in [
    "How does OSFI define a model in its model risk guideline?",
    "What is the maximum loan-to-value ratio for a HELOC?",
    "What is the minimum qualifying rate for uninsured mortgages?",
    "What does OSFI expect banks to do to manage cyber security risk?",
    "What is the best recipe for banana bread?",
]]

Q: How does OSFI define a model in its model risk guideline?
[ANSWERED] A model is defined as an application of theoretical, empirical, judgmental assumptions or statistical techniques, including AI/ML methods, which processes input data to generate results. [2]
   [2] E-23 › A. Overview › A.4 Key Terms › Model
   (112.37s)

Q: What is the maximum loan-to-value ratio for a HELOC?
[ANSWERED] According to the passages, OSFI expects the non-amortizing HELOC component of a residential mortgage to have a maximum authorized LTV ratio of less than or equal to 65 percent [2][4].
   [2] B-20 › II. Principles › Loan-to-Value (LTV) ratio › LTV Ratio and Loan Type › (iv) Home Equity Lines of Credit (HELOCs)
   [4] B-20-CLAR › Background
   (2.1s)

Q: What is the minimum qualifying rate for uninsured mortgages?
[ANSWERED] The minimum qualifying rate for uninsured mortgages is the greater of the contractual mortgage rate plus 2% or 5.25%. [1]
   [1] B-20-INFO › Risks to the financial system and the 

In [7]:
print("The full JSON response for one question:")
print(json.dumps(examples[1], indent=2)[:2500])

The full JSON response for one question:
{
  "question": "What is the maximum loan-to-value ratio for a HELOC?",
  "answer": "According to the passages, OSFI expects the non-amortizing HELOC component of a residential mortgage to have a maximum authorized LTV ratio of less than or equal to 65 percent [2][4].",
  "status": "answered",
  "citations": [
    2,
    4
  ],
  "sources": [
    {
      "n": 1,
      "doc_id": "B-20",
      "section": "II. Principles \u203a Loan-to-Value (LTV) ratio \u203a LTV Ratio and Loan Type \u203a (iv) Home Equity Lines of Credit (HELOCs)",
      "url": "https://www.osfi-bsif.gc.ca/en/guidance/guidance-library/residential-mortgage-underwriting-practices-procedures-guideline-2017",
      "chunk_id": "B-20-039",
      "relevance": 5.028,
      "cited": false
    },
    {
      "n": 2,
      "doc_id": "B-20",
      "section": "II. Principles \u203a Loan-to-Value (LTV) ratio \u203a LTV Ratio and Loan Type \u203a (iv) Home Equity Lines of Credit (HELOCs)",
   

In [8]:
print("Search only, without the LLM:")
hits = requests.post(f"{BASE}/search", json={"query": "GDS and TDS ratio calculation", "k": 3}).json()
for h in hits:
    print(f"  {h['rank']}. {h['doc_id']} › {h['section'][:70]} (score {h['score']:.2f})")

Search only, without the LLM:
  1. B-20 › II. Principles › Debt service coverage (score 5.38)
  2. B-20 › II. Principles › Debt service coverage (score 5.10)
  3. B-20-INFO › Risks to the financial system and the impact of Guideline B-20 › Risks (score 1.39)


## Step 6: Open the interactive API documentation

FastAPI automatically creates a web page that documents every endpoint and lets you try them. Run the next cell, then click the link it prints. On the page, open **POST /ask**, click **Try it out**, type a question, and click **Execute**.

In [9]:
try:
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8000, path="/docs")
except ImportError:
    print("Open http://localhost:8000/docs in your browser")

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Step 7: Save example responses for the README

In [10]:
out = os.path.join(PROJECT_DIR, "evaluation", "api_examples.json")
json.dump(examples, open(out, "w"), indent=2)
print("Saved", out)

Saved /content/drive/MyDrive/osfi_rag_project/evaluation/api_examples.json
